# Reversal Chart from Atomic Trades

Reversal candles are constructed from atomic (tick-level) trades instead of fixed time intervals. A new candle forms when the price reverses by a specified amount, making them useful for filtering out noise and focusing on significant price movements.

Two key parameters control candle formation:
- **min_height**: Minimum candle body height (in price ticks) required to close a candle.
- **min_reversal**: Minimum reversal distance (in price ticks) from the high/low to trigger a new candle.

This notebook demonstrates how to:
1. Fetch atomic trades for a symbol
2. Build reversal candles from those trades
3. Plot the resulting reversal candlestick chart

In [1]:
import binpan
binpan.__version__

'0.8.14'

In [2]:
!python -V

Python 3.13.12


## Create Symbol Object

Create a `Symbol` object for the LTCBTC trading pair with 5-minute candles. The `hours=5` parameter fetches data for the last 5 hours.

In [3]:
ltc = binpan.Symbol(symbol='ltcusdt',
                    tick_interval='1m',
                    time_zone='Europe/Madrid',
                    hours=5)

2026-03-05	 20:53:21     INFO get_candles_by_time_stamps -> symbol=LTCUSDT tick_interval=1m start=2026-03-05 15:52:00 end=2026-03-05 20:52:59 limit=300
2026-03-05 20:53:21     INFO [panzer.binance.public.spot] Inicializando BinancePublicClient(market=spot)
2026-03-05 20:53:22     INFO [panzer.binance.public.spot] Rate limiter inicializado: max_per_minute=6000 safety_ratio=0.90


## Fetch Atomic Trades

Fetch the individual (tick-level) trades for the time period covered by the Symbol object.

In [4]:
ltc.get_atomic_trades()

2026-03-05	 20:53:23     INFO Obteniendo atomic trades de LTCUSDT por tiempo: 2026-03-05 14:52:00.000000 -> 2026-03-05 19:51:00.000000


Requesting atomic trades between 2026-03-05 15:52:00 and 2026-03-05 20:51:00


2026-03-05	 20:53:24     INFO Rango de IDs atómicos descubierto para LTCUSDT: 566597965 -> 566663269 (estimado 65305 trades)
2026-03-05	 20:53:25     INFO Peticiones API atomic trades LTCUSDT: 1 (ID actual: 566598965)
2026-03-05	 20:53:25     INFO Peticiones API atomic trades LTCUSDT: 2 (ID actual: 566599965)
2026-03-05	 20:53:26     INFO Peticiones API atomic trades LTCUSDT: 3 (ID actual: 566600965)
2026-03-05	 20:53:26     INFO Peticiones API atomic trades LTCUSDT: 4 (ID actual: 566601965)
2026-03-05	 20:53:26     INFO Peticiones API atomic trades LTCUSDT: 5 (ID actual: 566602965)
2026-03-05	 20:53:26     INFO Peticiones API atomic trades LTCUSDT: 6 (ID actual: 566603965)
2026-03-05	 20:53:27     INFO Peticiones API atomic trades LTCUSDT: 7 (ID actual: 566604965)
2026-03-05	 20:53:27     INFO Peticiones API atomic trades LTCUSDT: 8 (ID actual: 566605965)
2026-03-05	 20:53:27     INFO Peticiones API atomic trades LTCUSDT: 9 (ID actual: 566606965)
2026-03-05	 20:53:28     INFO Peticion

,Trade Id,Price,Quantity,Quote quantity,Date,Timestamp,Buyer was maker,Best price match
LTCUSDT Europe/Madrid,,,,,,,,
0,566597965,56.52,0.092,5.19984,2026-03-05 15:52:00,1772722320130,False,True
1,566597966,56.52,0.091,5.14332,2026-03-05 15:52:00,1772722320130,False,True
2,566597967,56.52,0.178,10.06056,2026-03-05 15:52:00,1772722320130,False,True
3,566597968,56.52,0.093,5.25636,2026-03-05 15:52:00,1772722320130,False,True
4,566597969,56.53,0.178,10.06234,2026-03-05 15:52:00,1772722320130,False,True
...,...,...,...,...,...,...,...,...
65995,566663960,55.74,2.007,111.87018,2026-03-05 20:18:54,1772738334359,False,True
65996,566663961,55.74,0.180,10.03320,2026-03-05 20:18:54,1772738334680,False,True
65997,566663962,55.74,0.692,38.57208,2026-03-05 20:18:56,1772738336189,False,True


In [5]:
# Verify that Trade IDs are consecutive (no missing trades)
assert len(ltc.atomic_trades['Trade Id'].diff().value_counts()) == 1

## Build Reversal Candles

Convert the atomic trades into reversal candles. The default parameters are `min_height=7` and `min_reversal=4`.

In [6]:
# Build reversal candles from the atomic trade data
ltc.get_reversal_atomic_candles()

,Open,High,Low,Close,Quantity,Timestamp
Timestamp,,,,,,
2026-03-05 15:52:00.130000+01:00,56.52,56.53,56.31,56.36,494.890,1772722320130
2026-03-05 15:53:13.160000+01:00,56.36,56.45,56.36,56.40,207.179,1772722393160
2026-03-05 15:53:56.180000+01:00,56.40,56.41,56.32,56.37,324.098,1772722436180
2026-03-05 15:55:07.125000+01:00,56.37,56.45,56.35,56.40,597.064,1772722507125
2026-03-05 15:56:10.022000+01:00,56.40,56.40,56.33,56.38,108.196,1772722570022
...,...,...,...,...,...,...
2026-03-05 20:03:27.394000+01:00,55.51,55.69,55.50,55.64,1416.661,1772737407394
2026-03-05 20:10:15.842000+01:00,55.64,55.75,55.62,55.70,828.331,1772737815842
2026-03-05 20:13:32.113000+01:00,55.70,55.79,55.68,55.74,727.143,1772738012113


## Plot Reversal Candles

Plot the reversal candlestick chart. Here `min_height=10` (larger than default) will recompute the reversal candles.

In [7]:
ltc.plot_reversal(min_height=10, min_reversal=4, candles_ta_height_ratio=0.9, from_atomic=True)

'/home/nando/PycharmProjects/binpan_studio_dev/notebooks/last_plot.png'